## Análisis de carga docente

Este análisis tiene como objetivo evaluar la distribución de carga académica en docentes, identificando posibles casos de sobrecarga o subutilización.

Se realiza un proceso de limpieza, transformación y análisis de los datos para generar insights que apoyen la toma de decisiones.

In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import os #direcciones 
import unicodedata
import numpy as np
import datetime
import math
from unidecode import unidecode

In [ ]:
from difflib import get_close_matches

In [ ]:
#cargar datos 
datos = pd.read_excel('Oferta_09042024.xlsx', sheet_name='datos')
#eliminar tildes 
datos = datos.applymap(lambda x: unidecode(x) if isinstance(x, str) else x)
#columnas que se quedan 

columnas = [0,1,3,4,5,7,9,10,11,12,13,14,15,16,17,20,21,24,26,27,28]
data = datos.iloc[:, columnas]

## DATA

Base inicial

In [ ]:
#limpieza de datos 

#pasar a entero ID
data = data.dropna(subset=['Instructor (ID)'])
data['Instructor (ID)'] = data['Instructor (ID)'].astype(int)

data= data.drop_duplicates(subset=['Instructor (ID)','Hora de inicio','Hora final','Lunes','Martes','Miércoles','Jueves','Viernes','Sábado','Domingo'], keep='first')

#llenar nulos en categoria del tipo del evento 

for index, row in data.iterrows():
    if pd.isna(row['Categoría del tipo de evento']):
        data.at[index, 'Categoría del tipo de evento'] = 'Sin información'
    if pd.isna(row['Categoría del tipo de eventos (denom.)']):
        data.at[index, 'Categoría del tipo de eventos (denom.)'] = 'Sin información'


## Procesamiento de datos

Se realiza la limpieza y transformación de las variables necesarias para el análisis de carga docente.

In [ ]:
# Calcular las horas trabajadas 

columnas_hora = ['Hora de inicio', 'Hora final'] 
for col in columnas_hora:
    data[col] = data[col].apply(lambda x: x.strftime('%H:%M') if isinstance(x, datetime.time) else x)

# Convertir columnas de hora a tipo datetime
for col in columnas_hora:
    data[col] = pd.to_datetime(data[col])

# Calcular la diferencia entre las columnas 'Hora final' y 'Hora de inicio' en horas
data['Diferencia'] = (data['Hora final'] - data['Hora de inicio']).dt.seconds / 3600

# Redondear la columna 'Diferencia' hacia arriba y sobrescribir los valores
data['Diferencia'] = data['Diferencia'].apply(lambda x: math.ceil(x))

In [ ]:
#Dia de la semana 
def dia_trabajo(row):
    for col in data.columns[8:15]:  # Iterar sobre las columnas de los días de la semana
        if row[col]:  # Si hay clase ese día
            return col
    return "Otro"  # Si no hay clase en ningún día

data['Dia de trabajo'] = data.apply(dia_trabajo, axis=1)

In [ ]:
def clasificar_periodo(hora):
    if 0 <= hora < 6:
        return "Madrugada"
    elif 6 <= hora < 12:
        return "Mañana"
    elif 12 <= hora < 18:
        return "Tarde"
    elif 18 <= hora <= 23:
        return "Noche"
    else:
        return "Hora inválida"

# Convertir las columnas a formato datetime si no lo están ya
data['Hora de inicio'] = pd.to_datetime(data['Hora de inicio'])
data['Hora final'] = pd.to_datetime(data['Hora final'])

# Crear la nueva columna clasificando según la hora de inicio
data['Periodo del dia'] = data['Hora de inicio'].dt.hour.apply(clasificar_periodo)

## Data 1 

Base inforamción horas de clase según tipo de contrato

In [ ]:
#BASE DE DATOS 1 
colum=[0,1,2,4,5,15,16,17,18,19,20,21,22]
data1=data.iloc[:, colum]

data1['horas_semanales'] = np.where(data1['SDvPer.'] == 'CATE', 10,
                                 np.where(data1['SDvPer.'] == 'DOIN', 12,
                                          np.where(data1['SDvPer.'] == 'DOCE', 20, 0)))

## Data 6

Base comparacion de horas de clase, horas libres y diferencia de horas

In [ ]:
c=[4,5]
data6=data.iloc[:, c]


data6 = data6.groupby('Instructor (ID)').first().reset_index()

data6['horas_semanales'] = np.where(data6['SDvPer.'] == 'CATE', 10,
                                 np.where(data6['SDvPer.'] == 'DOIN', 12,
                                          np.where(data6['SDvPer.'] == 'DOCE', 20, 0)))

data6['horas_semestrales'] = data6['horas_semanales'] * 16

# Agrupar por 'Instructor (ID)' y sumar 'Diferencia'
suma_diferencia = data1.groupby('Instructor (ID)')['Diferencia'].sum().reset_index()

# Renombrar la columna de suma para hacer el merge
suma_diferencia.rename(columns={'Diferencia': 'horas 20241'}, inplace=True)

# Merge de data6 con suma_diferencia en base a 'Instructor (ID)'
data6 = pd.merge(data6, suma_diferencia, on='Instructor (ID)', how='left')

#horas libres 


data6['horas_libres'] = np.where(data6['horas_semanales'] > data6['horas 20241'], 'sí', 'no')
data6['diferencia_horas']=data6['horas_semanales']-data6['horas 20241']

## Analitica descriptiva

Se calcula el promedio de horas trabajas por los docentes según su tipo de contrato

In [ ]:
# Filtrar los datos donde la columna tipo de contrato sea docente investigador
filtro_doin = data6[data6['SDvPer.'] == 'DOIN']
promedio_doin = filtro_doin['diferencia_horas'].mean()
promedio_doin

# Filtrar los datos donde la columna tipo de contrato sea docente planta
filtro_doce = data6[data6['SDvPer.'] == 'DOCE']
promedio_doce = filtro_doce['diferencia_horas'].mean()
promedio_doce

# Filtrar los datos donde la columna tipo de contrato sea docente catedra
filtro_cate = data6[data6['SDvPer.'] == 'CATE']
promedio_cate = filtro_cate['diferencia_horas'].mean()
promedio_cate

Se calcula el porcentaje de docentes con sobrecarga horaria

#porcentaje de docentes con cobrecarga horaria
sobrecarga = data6[data6['diferencia_horas'] < 0]
conteo = sobrecarga['Instructor (ID)'].nunique()
porce=conteo/958
porce

### Distribución de horas semanales

Se analiza la distribución de horas asignadas para identificar posibles patrones de sobrecarga o subutilización.

In [ ]:
import matplotlib.pyplot as plt

plt.hist(data['horas_semanales'], bins=20)
plt.title("Distribución de horas semanales")
plt.xlabel("Horas")
plt.ylabel("Frecuencia")
plt.show()

### Horas por tipo de contrato

Se analiza cómo se distribuyen las horas según el tipo de contrato para identificar diferencias en la asignación.

In [ ]:
import seaborn as sns

sns.boxplot(x='tipo_contrato', y='horas_semanales', data=data)
plt.title("Horas semanales por tipo de contrato")
plt.xticks(rotation=45)
plt.show()

### Relación entre variables

Se presenta un análisis de correlación entre variables relevantes para identificar relaciones que puedan influir en la carga docente.

In [ ]:
orden_periodos = ['Mañana', 'Tarde', 'Noche']
data['Periodo del dia'] = pd.Categorical(data['Periodo del dia'], categories=orden_periodos, ordered=True)


orden_dias = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado']
data['Dia de trabajo'] = pd.Categorical(data['Dia de trabajo'], categories=orden_dias, ordered=True)

# Filtrar los datos para excluir la categoría 'Madrugada'
filtered_data = data[data['Periodo del dia'] != 'Madrugada']

# Crear la tabla pivote con los ejes ordenados
heatmap_data = filtered_data.pivot_table(
    index='Dia de trabajo', 
    columns='Periodo del dia', 
    values='Diferencia', 
    aggfunc='sum', 
    fill_value=0
)

# Crear el heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(heatmap_data, cmap='Blues', annot=True, fmt='.1f', linewidths=.5)

# Agregar títulos y etiquetas
plt.title('Horas por día de la semana')
plt.xlabel('SDvPer.')
plt.ylabel('Día de trabajo')

# Mostrar el gráfico
plt.show()

## Conclusiones

A partir del análisis realizado se identifican diferencias en la asignación de carga docente, evidenciando posibles casos de sobrecarga y subutilización.

El análisis permite identificar patrones relevantes según el tipo de contrato, lo que puede servir como base para la optimización en la asignación de recursos académicos.